# Gaussian Simulation — Improvement Boxplot (Part D: Evaluate on $\max(25,1.75k^*)$, Select on $\max(25,1.75k^*)+10$, $m=35$)

Variant of `gaussian_simulation_boxplot_K40.ipynb` (in `partC_gaussian_simulation_smoothed_peak_gap/`)
with the r* selection method swapped to the one validated on Part A/Part B
in `partD_quicktest/`:
- **Evaluation** (every reported rho, including the per-r boxplot groups
  and the r* group): computed only over $K = 2..\max(25, 1.75k^*)$ -- the
  reference paper's own K-range cap (Section 3.2).
- **Selection** of r*: `select_r_topK_spearman` (original top-m
  monotonic-decrease criterion) with $m=35$, over a slightly wider range
  $K = 2..(\max(25,1.75k^*)+10)$ -- the `+10` buffer gives the m=35 window
  headroom past the evaluation boundary while still overlapping
  substantially back into it.

The representative Case 1 setting uses true $K^*=10$.
(mean rho 0.233 there) -- this notebook checks whether that weakness
holds up with real replication (N_REPS=100) and a boxplot distribution,
rather than the single-seed N_REP=1 quicktest.


In [ ]:
# -- Core numerical stack --
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr

# -- Data generation, standardization, clustering, and external validity metrics --
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

# -- k-nearest-neighbor queries used to build the rNN affinity distance matrices --
from sklearn.neighbors import NearestNeighbors

# -- Plotting (final boxplot figures) --
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## Helper functions

In [ ]:
def simulate_gaussian_data(n_samples=2000, n_features=20, K_star=10,
                           cluster_std=1, random_state=1):
    """Generate one replicate of mixture-of-Gaussians data. Thin wrapper
    around sklearn's make_blobs: `random_state` differs per replicate in
    the main loop below (rep=0..N_REPS-1), so each of the 100 repetitions
    draws an independent dataset from the same generative model (K_star
    isotropic Gaussian clusters in n_features dimensions)."""
    X, y = make_blobs(n_samples=n_samples, n_features=n_features,
                      centers=K_star, cluster_std=cluster_std,
                      random_state=random_state)
    return X, y


def clustering_accuracy(y_true, y_pred):
    """Clustering accuracy via the Hungarian algorithm.

    Finds the best 1-to-1 matching between predicted cluster labels and
    true class labels (since predicted labels are arbitrary integers) by
    maximizing the matched point count with linear_sum_assignment, then
    reports that best count as a fraction of all points.
    """
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=np.int64)
    for i in range(y_pred.size):
        w[y_pred[i], y_true[i]] += 1
    row_ind, col_ind = linear_sum_assignment(w.max() - w)
    return w[row_ind, col_ind].sum() / y_pred.size


def rnn_affinity_distance(X, r, use_gaussian=False):
    """
    Build rNN affinity distance matrix (precomputed, zeros on diagonal).
    Hard (binary): distance = 0 when either point lists the other among its r nearest neighbors.
    Soft (Gaussian): Gaussian-weighted affinity converted to distance.

    Vectorized variant of the same construction used in
    gaussian_simulation.ipynb (there implemented with Python loops); this
    version uses fancy indexing since it is called many more times here
    (once per rep per r -- N_REPS * len(R_VALUES) calls in the main loop
    below, vs. only len(R_VALUES) calls in gaussian_simulation.ipynb).
    """
    n = X.shape[0]
    # Query r+1 neighbors and remove self so r counts other observations.
    # (Package-wide convention: r = number of NON-SELF neighbors.)
    nbrs = NearestNeighbors(n_neighbors=r + 1, algorithm='auto').fit(X)
    dists, idx = nbrs.kneighbors(X)
    dists, idx = dists[:, 1:], idx[:, 1:]

    A = np.zeros((n, n))
    rows = np.arange(n)[:, None]
    if use_gaussian:
        # Soft/Gaussian-weighted affinity: bandwidth sigma_i per point i is
        # its distance to its r-th (farthest of its r) neighbor, so the
        # weighting adapts to local density.
        sigma = dists[:, -1].copy()
        sigma[sigma == 0] = 1e-10   # guard against sigma=0 (coincident points)
        A[rows, idx] = np.exp(-(dists ** 2) / (2 * sigma[:, None] ** 2))
    else:
        # Hard/binary affinity: 1 if j is among i's r nearest neighbors.
        A[rows, idx] = 1.0

    A = np.maximum(A, A.T)   # symmetrise (OR rule)
    np.fill_diagonal(A, 0)
    a_min, a_max = A.min(), A.max()
    if a_max > a_min:
        A = (A - a_min) / (a_max - a_min)
    D = 1.0 - A
    np.fill_diagonal(D, 0)
    return D


def spearmanr_safe(a, b):
    """Spearman correlation, returns NaN on degenerate input."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    mask = ~np.isnan(a) & ~np.isnan(b)
    if mask.sum() < 2:
        return np.nan
    rho, _ = spearmanr(a[mask], b[mask])
    return float(rho)


def select_r_topK_spearman(sil_dict, r_values, m, n_k, threshold=0.95):
    '''
    Original r* selection criterion (Algorithm 1 in the manuscript). For
    each candidate r, take the observed silhouette curve's top-m largest-K
    values (`sil_dict[r]` sliced to the top m_eff positions of the K-sweep)
    and check how closely it follows a strictly DECREASING trend by
    computing its Spearman correlation against the ideal strictly
    decreasing sequence [m, m-1, ..., 1]. The rationale: once K exceeds the
    true number of clusters, a well-behaved validity score should keep
    decreasing as K grows further past K* (more splits = worse fit), so a
    silhouette curve that decreases cleanly over large K's is evidence that
    r is large enough to have "seen" the true cluster structure.

    Among r's whose criterion score clears `threshold`, picks the smallest
    such r (candidates are already in ascending order via `sorted(...)`,
    and `next(...)` returns the first one satisfying the condition) --
    i.e., the smallest neighborhood size that already shows a clean
    monotonic decrease, avoiding unnecessarily large r. If no r clears the
    threshold, falls back to whichever r has the single highest criterion
    score (best available, even if imperfect).
    '''
    avail = sorted(r for r in r_values if r in sil_dict)
    if not avail:
        return r_values[0]
    m_eff = max(2, min(m, n_k - 1))               # can't exceed the number of available K's
    top_idx = list(range(n_k - m_eff, n_k))         # indices of the top-m_eff largest K's
    ideal = np.arange(m_eff, 0, -1, dtype=float)    # ideal strictly-decreasing reference [m_eff, ..., 1]
    c_vals = {}
    for r in avail:
        obs = np.asarray(sil_dict[r], dtype=float)[top_idx]
        rho = spearmanr_safe(obs, ideal)
        c_vals[r] = rho if not np.isnan(rho) else -2.0   # sentinel below any valid rho so NaNs never win the fallback argmax
    r_star = next((r for r in avail if c_vals[r] >= threshold), None)
    if r_star is None:
        r_star = max(avail, key=lambda r: c_vals[r])
    return r_star


def max_K_eval(k_star):
    '''max(25, 1.75*k*) -- the reference paper's own K-range cap (Section
    3.2), used here as the EVALUATION scope.'''
    return int(max(25, round(1.75 * k_star)))


def max_K_select(k_star):
    '''max_K_eval(k*) + 10 -- a small buffer ADDED ONLY for r* selection.
    Widening the K-sweep window used for the top-m selection criterion
    (relative to the evaluation window) gives the m=35-point window enough
    headroom past the evaluation boundary, while still overlapping
    substantially back into the evaluation range -- see the memory note on
    r*-selection method design for why a buffer of 0 or of size m itself
    both perform worse than this intermediate +10 choice.'''
    return max_K_eval(k_star) + 10


## Settings

In [ ]:
K_STAR        = 10       # true number of Gaussian clusters (ground truth K*), matching gaussian_simulation.ipynb
N_REPS        = 100      # number of independent dataset replicates to average over
R_VALUES      = list(range(5, 135, 5))   # full grid: 5,10,...,130 (used for simulation & r* selection)
R_VALUES_PLOT = [5, 10, 30, 50, 100]     # subset of R_VALUES actually shown as boxplot groups in the figure

M         = 35     # window size for select_r_topK_spearman's top-m monotonic-decrease criterion
THRESHOLD = 0.95    # minimum Spearman rho (vs. the ideal decreasing sequence) required to accept an r

MAX_K_SELECT = max_K_select(K_STAR)                # K-range upper bound used for r* SELECTION (with +10 buffer)
MAX_K_EVAL   = max_K_eval(K_STAR)                  # K-range upper bound used for every reported/evaluated rho
K_TEST       = list(range(2, MAX_K_SELECT + 1))    # K sweep for clustering & r* selection window (the wider range)
EVAL_IDX     = [i for i, K in enumerate(K_TEST) if K <= MAX_K_EVAL]  # indices into K_TEST restricting to the narrower evaluation range

print(f'True K = {K_STAR},  reps = {N_REPS}')
print(f'r grid (simulation): {R_VALUES}')
print(f'r grid (plot):       {R_VALUES_PLOT}')
print(f'K sweep (select): {K_TEST[0]}..{K_TEST[-1]} ({len(K_TEST)} values), m={M}')
print(f'K eval scope:     {K_TEST[0]}..{MAX_K_EVAL} ({len(EVAL_IDX)} values)')


## Evaluation function

In [ ]:
def evaluate_one_rep(X, y_true, K_test, eval_idx, r, random_state=42,
                     labels_by_K=None, baseline=None):
    """
    For a single replicate and a single candidate r, cluster across every K
    in K_test (or reuse pre-computed `labels_by_K`, since K-means labels do
    not depend on r), build the Hard and Soft rNN affinity distance
    matrices for this r, and compute silhouette scores (ordinary + Hard +
    Soft) for every K.

    Returns both:
      - the Spearman rho (over the `eval_idx` slice of K_test) between each
        silhouette variant and each external metric (ACC/ARI/NMI) -- 9
        rho values total, used directly by the main loop to compute
        "improvement of rNN over ordinary Silhouette" per rep/per r; and
      - the raw arrays themselves (silhouette curves + external metric
        values for all of K_test, not just eval_idx), which the main loop
        needs at FULL K_TEST resolution to run select_r_topK_spearman
        (the r*-selection window is wider than the evaluation window).

    The `baseline` / `compute_baseline` machinery lets the caller skip
    recomputing the ordinary silhouette and the three external metrics on
    every call, since those don't depend on r and only need to be computed
    once per replicate (see the main loop, which passes `baseline=None` on
    the first r of each rep and the previously-computed baseline
    thereafter).

    Returns
    -------
    rho_orig_acc, rho_hard_acc, rho_soft_acc,
    rho_orig_ari, rho_hard_ari, rho_soft_ari,
    rho_orig_nmi, rho_hard_nmi, rho_soft_nmi,
    sil_orig (array), sil_hard (array), sil_soft (array),
    acc_vals (array), ari_vals (array), nmi_vals (array)
    """
    D_hard = rnn_affinity_distance(X, r=r, use_gaussian=False)
    D_soft = rnn_affinity_distance(X, r=r, use_gaussian=True)

    if labels_by_K is None:
        labels_by_K = {K: KMeans(n_clusters=K, random_state=random_state,
                                      n_init='auto').fit_predict(X)
                       for K in K_test}

    compute_baseline = baseline is None
    if compute_baseline:
        sil_orig, acc_vals, ari_vals, nmi_vals = [], [], [], []
    else:
        # Reuse the ordinary-silhouette / external-metric arrays computed
        # on an earlier r for this same replicate -- they are independent
        # of r, so recomputing them for every r would be wasted work.
        sil_orig, acc_vals, ari_vals, nmi_vals = [np.asarray(v) for v in baseline]
    sil_hard, sil_soft = [], []

    for K in K_test:
        labels = labels_by_K[K]
        if len(np.unique(labels)) > 1:
            if compute_baseline:
                sil_orig.append(silhouette_score(X, labels, metric='euclidean'))
            sil_hard.append(silhouette_score(D_hard, labels, metric='precomputed'))
            sil_soft.append(silhouette_score(D_soft, labels, metric='precomputed'))
        else:
            # Degenerate clustering (single unique label) -- silhouette
            # undefined; record NaN, masked out later by spearmanr_safe.
            if compute_baseline:
                sil_orig.append(np.nan)
            sil_hard.append(np.nan)
            sil_soft.append(np.nan)
        if compute_baseline:
            acc_vals.append(clustering_accuracy(y_true, labels))
            ari_vals.append(adjusted_rand_score(y_true, labels))
            nmi_vals.append(normalized_mutual_info_score(y_true, labels))

    sil_orig = np.asarray(sil_orig)
    sil_hard = np.array(sil_hard)
    sil_soft = np.array(sil_soft)
    acc_vals = np.asarray(acc_vals)
    ari_vals = np.asarray(ari_vals)
    nmi_vals = np.asarray(nmi_vals)

    def _rho(sil, ref):
        # Spearman correlation restricted to the evaluation-range indices
        # (eval_idx), i.e. K = 2..MAX_K_EVAL, even though sil/ref were
        # computed over the wider K_test (2..MAX_K_SELECT).
        sil_e, ref_e = sil[eval_idx], ref[eval_idx]
        mask = ~np.isnan(sil_e) & ~np.isnan(ref_e)
        return spearmanr(sil_e[mask], ref_e[mask])[0] if mask.sum() >= 2 else np.nan

    return (
        _rho(sil_orig, acc_vals), _rho(sil_hard, acc_vals), _rho(sil_soft, acc_vals),
        _rho(sil_orig, ari_vals), _rho(sil_hard, ari_vals), _rho(sil_soft, ari_vals),
        _rho(sil_orig, nmi_vals), _rho(sil_hard, nmi_vals), _rho(sil_soft, nmi_vals),
        sil_orig, sil_hard, sil_soft, acc_vals, ari_vals, nmi_vals,
    )

## Main repetition loop

In [ ]:
# Main replication loop: for each of N_REPS independent Gaussian-mixture
# replicates, this loop
#   (1) generates a fresh dataset and standardizes it,
#   (2) clusters it once per K (labels don't depend on r, so they're reused
#       across every candidate r below),
#   (3) for every r in R_VALUES, computes the Spearman-rho IMPROVEMENT of
#       the rNN-based Silhouette (Hard and Soft) over the ordinary
#       Silhouette, against each of ACC/ARI/NMI, and
#   (4) additionally selects r* per replicate (separately for Hard and
#       Soft) via select_r_topK_spearman, and records that same
#       improvement for the r*-selected curve.
# The resulting `all_reps_diff[r][metric][variant]` holds N_REPS
# improvement values per (r, metric, variant) combination -- exactly the
# distributions drawn as boxplots later in this notebook.
from collections import Counter

all_reps_diff = {
    r: {'acc': {'hard': [], 'soft': []},
        'ari': {'hard': [], 'soft': []},
        'nmi': {'hard': [], 'soft': []}}
    for r in R_VALUES
}
all_reps_diff['r_star'] = {'acc': {'hard': [], 'soft': []},
                            'ari': {'hard': [], 'soft': []},
                            'nmi': {'hard': [], 'soft': []}}
r_star_log = {'hard': [], 'soft': []}   # selected r* per rep

for rep in range(N_REPS):
    if (rep + 1) % 10 == 0:
        print(f'Rep {rep+1}/{N_REPS}')

    # Each replicate uses rep as its random_state, so the N_REPS datasets
    # are independent draws from the same generative model, then
    # standardized (mean 0 / unit variance per feature) before clustering.
    X_rep, y_rep = simulate_gaussian_data(
        n_samples=2000, n_features=20, K_star=K_STAR, random_state=rep)
    X_rep = StandardScaler().fit_transform(X_rep)

    # K-means labels and ordinary/reference metrics do not depend on r.
    # Compute them once per repetition and reuse them for all r candidates.
    labels_by_K = {K: KMeans(n_clusters=K, random_state=rep,
                                  n_init='auto').fit_predict(X_rep)
                   for K in K_TEST}
    baseline = None

    sil_hard_by_r = {}
    sil_soft_by_r = {}
    sil_orig_rep = acc_vals_rep = ari_vals_rep = nmi_vals_rep = None

    for r in R_VALUES:
        (rho_orig_acc, rho_hard_acc, rho_soft_acc,
         rho_orig_ari, rho_hard_ari, rho_soft_ari,
         rho_orig_nmi, rho_hard_nmi, rho_soft_nmi,
         sil_orig, sil_hard, sil_soft, acc_vals, ari_vals, nmi_vals) = evaluate_one_rep(
            X_rep, y_rep, K_TEST, EVAL_IDX, r=r, random_state=rep,
            labels_by_K=labels_by_K, baseline=baseline)

        # "Improvement" = rho(rNN-based silhouette, external metric) minus
        # rho(ordinary silhouette, external metric), for this rep and this r.
        all_reps_diff[r]['acc']['hard'].append(rho_hard_acc - rho_orig_acc)
        all_reps_diff[r]['acc']['soft'].append(rho_soft_acc - rho_orig_acc)
        all_reps_diff[r]['ari']['hard'].append(rho_hard_ari - rho_orig_ari)
        all_reps_diff[r]['ari']['soft'].append(rho_soft_ari - rho_orig_ari)
        all_reps_diff[r]['nmi']['hard'].append(rho_hard_nmi - rho_orig_nmi)
        all_reps_diff[r]['nmi']['soft'].append(rho_soft_nmi - rho_orig_nmi)

        sil_hard_by_r[r] = sil_hard
        sil_soft_by_r[r] = sil_soft
        # sil_orig / acc / ari / nmi don't depend on r — capture once
        if sil_orig_rep is None:
            sil_orig_rep = sil_orig
            acc_vals_rep  = acc_vals
            ari_vals_rep  = ari_vals
            nmi_vals_rep  = nmi_vals
            baseline = (sil_orig_rep, acc_vals_rep, ari_vals_rep, nmi_vals_rep)

    # Baseline (ordinary-silhouette) rho's for this rep, evaluated once
    # over EVAL_IDX -- used below as the reference the r*-selected curve's
    # improvement is measured against.
    rho_orig_acc_base = spearmanr_safe(sil_orig_rep[EVAL_IDX], acc_vals_rep[EVAL_IDX])
    rho_orig_ari_base = spearmanr_safe(sil_orig_rep[EVAL_IDX], ari_vals_rep[EVAL_IDX])
    rho_orig_nmi_base = spearmanr_safe(sil_orig_rep[EVAL_IDX], nmi_vals_rep[EVAL_IDX])

    # -- Select r* per rep: select_r_topK_spearman, m=35, over the WIDE K_TEST --
    # Selection is unsupervised w.r.t. the external metrics (it only looks
    # at the silhouette curves' own shape), so the same r* is used for all
    # three external metrics within a given rep/variant.
    r_star_h = select_r_topK_spearman(sil_hard_by_r, R_VALUES, m=M, n_k=len(K_TEST), threshold=THRESHOLD)
    r_star_s = select_r_topK_spearman(sil_soft_by_r, R_VALUES, m=M, n_k=len(K_TEST), threshold=THRESHOLD)
    r_star_log['hard'].append(r_star_h)
    r_star_log['soft'].append(r_star_s)

    all_reps_diff['r_star']['acc']['hard'].append(
        spearmanr_safe(sil_hard_by_r[r_star_h][EVAL_IDX], acc_vals_rep[EVAL_IDX]) - rho_orig_acc_base)
    all_reps_diff['r_star']['acc']['soft'].append(
        spearmanr_safe(sil_soft_by_r[r_star_s][EVAL_IDX], acc_vals_rep[EVAL_IDX]) - rho_orig_acc_base)
    all_reps_diff['r_star']['ari']['hard'].append(
        spearmanr_safe(sil_hard_by_r[r_star_h][EVAL_IDX], ari_vals_rep[EVAL_IDX]) - rho_orig_ari_base)
    all_reps_diff['r_star']['ari']['soft'].append(
        spearmanr_safe(sil_soft_by_r[r_star_s][EVAL_IDX], ari_vals_rep[EVAL_IDX]) - rho_orig_ari_base)
    all_reps_diff['r_star']['nmi']['hard'].append(
        spearmanr_safe(sil_hard_by_r[r_star_h][EVAL_IDX], nmi_vals_rep[EVAL_IDX]) - rho_orig_nmi_base)
    all_reps_diff['r_star']['nmi']['soft'].append(
        spearmanr_safe(sil_soft_by_r[r_star_s][EVAL_IDX], nmi_vals_rep[EVAL_IDX]) - rho_orig_nmi_base)

print('\nAll repetitions completed!')
print('\nr* selection frequency:')
for variant in ['hard', 'soft']:
    counts = Counter(r_star_log[variant])
    print(f'  {variant}: ' + ', '.join(f'r={r}: {counts[r]}x' for r in sorted(counts)))

# -- Summary: mean +/- std improvement across all N_REPS reps, for every
# fixed r and for the r*-selected curve. This is the numeric backbone of
# the boxplots produced later in this notebook.
print()
for r in R_VALUES:
    d_ha = np.array(all_reps_diff[r]['acc']['hard'])
    d_sa = np.array(all_reps_diff[r]['acc']['soft'])
    d_hr = np.array(all_reps_diff[r]['ari']['hard'])
    d_sr = np.array(all_reps_diff[r]['ari']['soft'])
    d_hn = np.array(all_reps_diff[r]['nmi']['hard'])
    d_sn = np.array(all_reps_diff[r]['nmi']['soft'])
    print(f'  r={r:3d}  '
          f'Hard(ACC): {d_ha.mean():.4f}±{d_ha.std():.4f}  '
          f'Soft(ACC): {d_sa.mean():.4f}±{d_sa.std():.4f}  '
          f'Hard(ARI): {d_hr.mean():.4f}±{d_hr.std():.4f}  '
          f'Soft(ARI): {d_sr.mean():.4f}±{d_sr.std():.4f}  '
          f'Hard(NMI): {d_hn.mean():.4f}±{d_hn.std():.4f}  '
          f'Soft(NMI): {d_sn.mean():.4f}±{d_sn.std():.4f}')

d_ha = np.array(all_reps_diff['r_star']['acc']['hard'])
d_sa = np.array(all_reps_diff['r_star']['acc']['soft'])
d_hr = np.array(all_reps_diff['r_star']['ari']['hard'])
d_sr = np.array(all_reps_diff['r_star']['ari']['soft'])
d_hn = np.array(all_reps_diff['r_star']['nmi']['hard'])
d_sn = np.array(all_reps_diff['r_star']['nmi']['soft'])
print(f'  r*  Hard(ACC): {d_ha.mean():.4f}±{d_ha.std():.4f}  '
      f'Soft(ACC): {d_sa.mean():.4f}±{d_sa.std():.4f}  '
      f'Hard(ARI): {d_hr.mean():.4f}±{d_hr.std():.4f}  '
      f'Soft(ARI): {d_sr.mean():.4f}±{d_sr.std():.4f}  '
      f'Hard(NMI): {d_hn.mean():.4f}±{d_hn.std():.4f}  '
      f'Soft(NMI): {d_sn.mean():.4f}±{d_sn.std():.4f}')

In [ ]:
# Cache the full 100-replicate results (and the per-rep selected r* log) so
# they can be reloaded below without rerunning the (expensive) main loop,
# and so combined_boxplot_figure.ipynb can reuse this exact pickle for the
# bottom panel of its combined figure.
import pickle, os

os.makedirs('data', exist_ok=True)
with open('data/gaussian_simulation_boxplot_paperkmax_buffer10_nonself_results.pkl', 'wb') as f:
    pickle.dump({
        'all_reps_diff':  all_reps_diff,
        'R_VALUES':       R_VALUES,
        'R_VALUES_PLOT':  R_VALUES_PLOT,
        'K_TEST':         K_TEST,
        'K_STAR':         K_STAR,
        'N_REPS':         N_REPS,
        'MAX_K_SELECT':   MAX_K_SELECT,
        'MAX_K_EVAL':     MAX_K_EVAL,
        'M':              M,
        'r_star_log':     r_star_log,
    }, f)
print('Results saved to data/gaussian_simulation_boxplot_paperkmax_buffer10_nonself_results.pkl')

## Boxplot

## Load saved results (skip the loop if data already exists)

In [ ]:
# Reload the cached 100-rep results (lets the rest of the notebook -- the
# frequency table and boxplot cells below -- be re-run without repeating
# the expensive main loop above).
import pickle
from collections import Counter

with open('data/gaussian_simulation_boxplot_paperkmax_buffer10_nonself_results.pkl', 'rb') as f:
    saved = pickle.load(f)

all_reps_diff = saved['all_reps_diff']
R_VALUES      = saved['R_VALUES']
R_VALUES_PLOT = saved.get('R_VALUES_PLOT', [5, 10, 30, 50, 100])
K_TEST        = saved['K_TEST']
K_STAR        = saved['K_STAR']
N_REPS        = saved['N_REPS']
MAX_K_SELECT  = saved.get('MAX_K_SELECT')
MAX_K_EVAL    = saved.get('MAX_K_EVAL')
M             = saved.get('M', 35)
r_star_log    = saved.get('r_star_log', {'hard': [], 'soft': []})

print(f'Loaded: K_STAR={K_STAR}, N_REPS={N_REPS}')
print(f'r grid (simulation): {R_VALUES}')
print(f'r grid (plot):       {R_VALUES_PLOT}')
print(f'K sweep (select): {K_TEST}')
print(f'max_K_select={MAX_K_SELECT}  max_K_eval={MAX_K_EVAL}  m={M}')
if any(r_star_log[v] for v in ('hard', 'soft')):
    print('\nr* selection frequency:')
    for variant in ['hard', 'soft']:
        counts = Counter(r_star_log[variant])
        print(f'  {variant}: ' + ', '.join(f'r={r}: {counts[r]}x' for r in sorted(counts)))

In [ ]:
# -- Frequency table of selected r* values --
# Shows, across the 100 replicates, how often each candidate r in R_VALUES
# was the one picked by select_r_topK_spearman as r* (separately for the
# Hard and Soft variants). Useful for sanity-checking that r* selection is
# concentrating on a narrow, sensible band of r rather than scattering
# widely across the grid.
from collections import Counter

variants   = ['hard', 'soft']
var_labels = {'hard': 'Neighborhood Hard', 'soft': 'Neighborhood Soft'}
col_w = 12

print(f"{'r':>6}  " + "  ".join(f"{var_labels[v]:>{col_w}}" for v in variants))
print("-" * (8 + (col_w + 2) * len(variants)))
for r in R_VALUES:
    row = f"{r:>6}  "
    for v in variants:
        counts = Counter(r_star_log[v])
        n      = counts.get(r, 0)
        pct    = 100 * n / len(r_star_log[v]) if r_star_log[v] else 0
        row   += f"{f'{n} ({pct:.0f}%)':>{col_w}}  "
    print(row)
print("-" * (8 + (col_w + 2) * len(variants)))
total_row = f"{'Total':>6}  "
for v in variants:
    total_row += f"{len(r_star_log[v]):>{col_w}}  "
print(total_row)

In [ ]:
def plot_one_panel(ax, all_reps_diff, r_values, ref_key, ext_label):
    """Draw one boxplot panel (ACC / ARI / NMI) onto ax: for each r in
    r_values, plus a final r* group, draw a pair of boxplots (Hard, Soft)
    of the per-rep Spearman-rho improvement over the ordinary silhouette
    (all N_REPS values, from all_reps_diff[r or 'r_star'][ref_key][variant]).
    The r* group is visually distinguished (hatched, higher alpha, and set
    off by a vertical separator + red box) since it represents the
    proposed method's actual output rather than an illustrative fixed r."""
    variants = ['hard', 'soft']
    colors   = {'hard': 'steelblue', 'soft': 'darkorange'}
    labels_legend = {
        'hard': 'Silhouette Score (Neighborhood Hard)',
        'soft': 'Silhouette Score (Neighborhood Soft)',
    }

    width = 0.28
    offsets_rel = [-width / 2, width / 2]

    r_star_keys   = ['r_star']
    r_star_labels = [r'$r^*$']

    plot_keys   = list(r_values) + r_star_keys
    plot_labels = [f'r = {r}' for r in r_values] + r_star_labels

    x_ticks, x_tick_labels, proxy_patches = [], [], {}

    for i, (rkey, rlabel) in enumerate(zip(plot_keys, plot_labels)):
        center = i * (2 * width + 0.5)
        is_r_star = rkey in r_star_keys
        for variant, offset in zip(variants, offsets_rel):
            data = all_reps_diff[rkey][ref_key][variant]
            pos  = center + offset
            bp = ax.boxplot(
                [data], positions=[pos], widths=width * 0.9,
                patch_artist=True,
                medianprops=dict(color='black', linewidth=2),
                manage_ticks=False,
            )
            patch = bp['boxes'][0]
            patch.set_facecolor(colors[variant])
            patch.set_alpha(0.85 if is_r_star else 0.65)
            if is_r_star:
                patch.set_hatch('///')
            if variant not in proxy_patches:
                proxy_patches[variant] = patch
        x_ticks.append(center)
        x_tick_labels.append(rlabel)

    # Vertical separator before the r* group
    sep_x = (len(r_values) - 1) * (2 * width + 0.5) + (2 * width + 0.5) * 0.75
    ax.axvline(sep_x, color='gray', linestyle=':', linewidth=1.5, alpha=0.6)

    # Highlight the r* group with a red-edged box
    pad     = width + 0.3
    left_x  = x_ticks[len(r_values)] - pad
    right_x = x_ticks[-1] + pad
    ax.axvspan(left_x, right_x, facecolor='none', edgecolor='red',
              linewidth=2.5, zorder=10)

    ax.axhline(0, color='grey', linewidth=1, linestyle='--')
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_tick_labels, fontsize=20)
    ax.tick_params(axis='y', labelsize=16)
    ax.set_xlabel('')
    ax.set_ylabel(
        f'Difference in Spearman $\\rho$ with {ext_label}',
        fontsize=18)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_title(ext_label, fontsize=29, fontweight='bold')

    return proxy_patches, labels_legend


def plot_diff_boxplot_single(all_reps_diff, r_values_plot, K_star, n_reps,
                             ref_key, ext_label, file_tag):
    """Draw and save a single standalone panel for one external measure
    (ACC / ARI / NMI), including the r* box-group (smoothed peak-gap)."""
    n_groups = len(r_values_plot) + 1
    fig, ax = plt.subplots(1, 1, figsize=(1.4 * n_groups + 2, 7))

    proxy_patches, labels_legend = plot_one_panel(
        ax, all_reps_diff, r_values_plot, ref_key, ext_label)

    fig.suptitle(
        f'Simulated Gaussian Dataset: Improvement of $r$NN Affinity Silhouette over Silhouette ({ext_label})\n'
        f'(True $\\mathbf{{K}} = {K_star}$, {n_reps} reps, eval $k_{{max}}$=max(25,1.75$k^*$), select +10, m=35)',
        fontsize=24, fontweight='bold',
    )

    handles = [proxy_patches[v] for v in ('hard', 'soft')]
    labels  = [labels_legend[v] for v in ('hard', 'soft')]
    fig.legend(
        handles, labels,
        fontsize=20,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.1),
        ncol=2,
        framealpha=0.9,
    )

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.24)
    import os
    os.makedirs('figures/smoothed_peak_gap', exist_ok=True)
    plt.savefig(f'figures/smoothed_peak_gap/SimGa_diff_boxplot_{file_tag}.pdf', bbox_inches='tight')
    plt.savefig(f'figures/smoothed_peak_gap/SimGa_diff_boxplot_{file_tag}.png', dpi=150, bbox_inches='tight')
    plt.show()


# Produce one standalone figure per external metric (ACC, ARI, NMI), each
# showing R_VALUES_PLOT boxplot groups plus the r* group.
plot_diff_boxplot_single(all_reps_diff, R_VALUES_PLOT, K_STAR, N_REPS, 'acc', 'ACC', 'acc_paperkmax_buffer10')
plot_diff_boxplot_single(all_reps_diff, R_VALUES_PLOT, K_STAR, N_REPS, 'ari', 'ARI', 'ari_paperkmax_buffer10')
plot_diff_boxplot_single(all_reps_diff, R_VALUES_PLOT, K_STAR, N_REPS, 'nmi', 'NMI', 'nmi_paperkmax_buffer10')